# Customer Segmentation using K-Means Clustering and PCA

**Dataset:** Mall Customer Segmentation Dataset (Kaggle)

**Objective:** Segment mall customers into groups based on annual income and spending behavior using K-Means Clustering, and visualize the clusters in 2D using PCA.


## Task 1: Data Understanding

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

sns.set(style="whitegrid")
%matplotlib inline


### 1.1 Load the dataset

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/gakudo-ai/open-datasets/refs/heads/main/Mall_Customers.csv"
df = pd.read_csv(url)
df.head()

### 1.2 Display the first five records

In [ ]:
df.head()

### 1.3 Identify numerical and categorical features

In [ ]:
numerical_features = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = df.select_dtypes(include=['object']).columns.tolist()

print("Numerical Features:", numerical_features)
print("Categorical Features:", categorical_features)


### 1.4 Dataset information and summary statistics

In [ ]:
df.info()


In [ ]:
df.describe(include='all')


## Task 2: Data Preprocessing

### 2.1 Check for missing values

In [ ]:
df.isnull().sum()


### 2.2 Remove unnecessary columns (e.g., CustomerID)

In [ ]:
df_processed = df.drop(columns=['CustomerID'])
df_processed.head()


### 2.3 Encode categorical variables (Gender)

In [ ]:
le = LabelEncoder()
df_processed['Gender'] = le.fit_transform(df_processed['Gender'])  # Male=1, Female=0
df_processed.head()


### 2.4 Standardize the numerical features

In [ ]:
features_to_scale = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']

scaler = StandardScaler()
scaled_values = scaler.fit_transform(df_processed[features_to_scale])

scaled_df = pd.DataFrame(scaled_values, columns=features_to_scale)
scaled_df['Gender'] = df_processed['Gender'].values
scaled_df.head()


## Task 3: Model Development

### 3.1 Elbow Method to determine optimal K

In [ ]:
X = scaled_df[['Annual Income (k$)', 'Spending Score (1-100)']].values

wcss = []
K_range = range(1, 11)
for k in K_range:
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    kmeans.fit(X)
    wcss.append(kmeans.inertia_)


### 3.2 Train K-Means with the selected K

In [ ]:
optimal_k = 5  # chosen from the elbow curve below

kmeans = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X)


### 3.3 Assign cluster labels to each customer

In [ ]:
df_processed['Cluster'] = cluster_labels
df_processed.head()


### 3.4 Apply PCA to reduce the dataset to 2 principal components

In [ ]:
pca_features = scaled_df.copy()

pca = PCA(n_components=2)
pca_result = pca.fit_transform(pca_features)

pca_df = pd.DataFrame(pca_result, columns=['PC1', 'PC2'])
pca_df['Cluster'] = cluster_labels

print("Explained variance ratio:", pca.explained_variance_ratio_)
pca_df.head()


## Task 4: Visualization and Evaluation

### 4.1 Elbow Curve

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(K_range, wcss, marker='o')
plt.title('Elbow Method for Optimal K')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('WCSS (Within-Cluster Sum of Squares)')
plt.xticks(list(K_range))
plt.show()


### 4.2 Scatter plot showing customer clusters (Income vs Spending Score)

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(x=df_processed['Annual Income (k$)'], y=df_processed['Spending Score (1-100)'],
                hue=df_processed['Cluster'], palette='tab10', s=70)
plt.title('Customer Segments (Annual Income vs Spending Score)')
plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.legend(title='Cluster')
plt.show()


### 4.3 PCA visualization with cluster labels

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(x=pca_df['PC1'], y=pca_df['PC2'], hue=pca_df['Cluster'], palette='tab10', s=70)
plt.title('Customer Clusters Visualized via PCA (2 Components)')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend(title='Cluster')
plt.show()


### Observations

1. **Optimal number of clusters:** The elbow curve shows the within-cluster sum of squares (WCSS) dropping
   sharply up to K=5, after which the improvement flattens out. This "elbow" at K=5 indicates that 5 clusters
   capture the natural grouping in the data without overfitting to noise.

2. **How PCA helps visualize high-dimensional data:** The dataset has multiple standardized features
   (Age, Income, Spending Score, Gender). PCA compresses this into 2 principal components that capture the
   maximum variance in the data, making it possible to plot and visually inspect cluster separation on a
   single 2D scatter plot — something not possible directly with 4+ dimensions.

3. **Characteristics of the identified customer groups:** The clusters generally separate into groups such as
   high-income/high-spending customers (prime marketing targets), high-income/low-spending customers
   (price-conscious, potential upsell targets), low-income/high-spending customers (value-driven impulse
   spenders), low-income/low-spending customers (low-priority segment), and an average-income/average-spending
   middle segment.

4. **Business relevance:** These segments allow the mall's marketing team to design targeted campaigns —
   for example, loyalty rewards for high-spenders and discount-driven promotions for price-sensitive segments —
   rather than applying a single generic marketing strategy to all customers.


## Task 5: Conclusion

This project applied K-Means Clustering to segment mall customers based on their annual income and spending
score, identifying five distinct customer groups after determining the optimal cluster count using the Elbow
Method. Principal Component Analysis was used to reduce the feature space to two dimensions, enabling clear
visualization of cluster separation. The resulting segments — ranging from high-income high-spenders to
budget-conscious shoppers — provide actionable insight for the mall's marketing team to design targeted
campaigns rather than a one-size-fits-all approach, improving marketing ROI and customer engagement.

A key limitation of K-Means is that it requires the number of clusters (K) to be specified in advance and
assumes clusters are roughly spherical and equally sized, which may not always reflect real customer behavior.
On the other hand, PCA's main advantage is that it reduces dimensionality while retaining most of the variance
in the data, making patterns easier to visualize and interpret without significant loss of information.
